# Example 07: Multimodal Input — Audio

Send an audio file alongside a text prompt.

This notebook generates a short test `.wav` file using the Python standard library
so you can run it without uploading anything. Replace `AUDIO_FILE` with the path to
your own recording if you want to test real audio.

**Note:** Audio support depends on the model variant. `gemma-4-E2B-it.litertlm` may not
include audio capabilities. Use a multimodal-specific variant if available.

In [ ]:
!pip install litert-lm-api-nightly litert-lm -q

In [ ]:
import subprocess
subprocess.run([
    "curl", "-L",
    "https://huggingface.co/litert-community/gemma-4-E2B-it-litert-lm/resolve/main/gemma-4-E2B-it.litertlm?download=true",
    "-o", "/content/gemma-4-E2B-it.litertlm"
], check=True)

In [ ]:
# Generate a minimal WAV file for testing (1 second of silence)
import struct, wave

AUDIO_FILE = "/content/test_audio.wav"
sample_rate = 16000
num_samples = sample_rate  # 1 second

with wave.open(AUDIO_FILE, "w") as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)  # 16-bit
    wf.setframerate(sample_rate)
    wf.writeframes(struct.pack("<" + "h" * num_samples, *([0] * num_samples)))

print(f"Test audio file created: {AUDIO_FILE}")
print("Replace AUDIO_FILE with a real recording path to test transcription.")

In [ ]:
import litert_lm

MODEL_PATH = "/content/gemma-4-E2B-it.litertlm"

litert_lm.set_min_log_severity(litert_lm.LogSeverity.ERROR)

with litert_lm.Engine(
    MODEL_PATH,
    audio_backend=litert_lm.Backend.CPU,
) as engine:
    with engine.create_conversation() as conversation:
        user_message = {
            "role": "user",
            "content": [
                {"type": "audio", "path": AUDIO_FILE},
                {"type": "text", "text": "Transcribe and summarise this audio."},
            ],
        }
        response = conversation.send_message(user_message)
        print(response["content"][0]["text"])